# Curadoria — Pipeline NOTICIAS_REAIS (RSS)

Lê todos os CSVs raw do pipeline `pipeline_noticias_reais/raw/`,
acumula o histórico, aplica limpeza e padronização e salva um CSV curated com timestamp.

**Operações desta camada:**
- Remover HTML de `titulo` e `resumo`
- Filtrar BBC BRASIL por termos políticos (categoria GERAL → filtrar por relevância)
- Padronizar `data_publicacao` de RFC para `YYYY-MM-DD`
- Remover duplicatas por `link`
- Montar `texto_principal` como `titulo` + separador + `resumo` limpo
- Adicionar colunas do schema obrigatório curated

## Bibliotecas

In [1]:
import re
import uuid
import pandas as pd

from datetime import datetime
from email.utils import parsedate_to_datetime
from pathlib import Path
from html.parser import HTMLParser

## Configuração

In [2]:
NOME_PIPELINE = "pipeline_noticias_reais"

PASTA_RAW     = Path(f"../dados/{NOME_PIPELINE}/raw")
PASTA_CURATED = Path(f"../dados/{NOME_PIPELINE}/curated")
PASTA_CURATED.mkdir(parents=True, exist_ok=True)

print(f"Raw:     {PASTA_RAW}")
print(f"Curated: {PASTA_CURATED}")

# Termos políticos para filtrar portais com categoria GERAL (ex: BBC, UOL)
TERMOS_POLITICOS = [
    "presidente", "governo", "congresso", "senado", "câmara", "deputado", "senador",
    "ministro", "ministério", "partido", "eleição", "eleições", "candidato",
    "stf", "supremo", "tse", "lula", "bolsonaro", "moraes", "pt", "pl ",
    "reforma", "votação", "plenário", "emenda", "orçamento", "imposto",
    "política", "político", "politica", "politico",
    "anistia", "pix", "inss", "previdência", "legislativo", "executivo", "judiciário",
    "veto", "decreto", "lei ", "pec", "mp ", "medida provisória",
]

# Limites de tamanho para montagem do texto_principal
RESUMO_MAX_CHARS  = 400   # resumo acima disso é artigo completo → usa só título
TEXTO_MIN_CHARS   = 30    # descartar textos muito curtos
TEXTO_LONGO_CHARS = 500   # marcar como longo (flag, não descarta)

Raw:     ..\dados\pipeline_noticias_reais\raw
Curated: ..\dados\pipeline_noticias_reais\curated


## Funções utilitárias

In [3]:
class _StripHTML(HTMLParser):
    def __init__(self):
        super().__init__()
        self._partes = []

    def handle_data(self, data):
        self._partes.append(data)

    def get_text(self):
        return " ".join(self._partes)


def remover_html(texto: str) -> str:
    if not isinstance(texto, str) or not texto.strip():
        return ""
    parser = _StripHTML()
    parser.feed(texto)
    limpo = parser.get_text()
    limpo = re.sub(r"\s+", " ", limpo).strip()
    return limpo


def padronizar_data_rfc(valor) -> str:
    """
    Converte data RFC 2822 para YYYY-MM-DD.
    Exemplo: 'Sat, 16 May 2026 10:15:00 -0300' → '2026-05-16'
    Fallback para outros formatos comuns.
    """
    if not isinstance(valor, str) or not valor.strip():
        return ""
    try:
        return parsedate_to_datetime(valor.strip()).strftime("%Y-%m-%d")
    except Exception:
        pass
    formatos = [
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d",
        "%d/%m/%Y",
    ]
    for fmt in formatos:
        try:
            return datetime.strptime(valor.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return ""


def contem_termos_politicos(texto: str, termos: list) -> bool:
    """Verifica se o texto contém ao menos um termo político."""
    texto_lower = texto.lower()
    return any(t in texto_lower for t in termos)


print("Funções utilitárias definidas.")

Funções utilitárias definidas.


## Leitura de todos os CSVs raw

In [4]:
arquivos_raw = sorted(PASTA_RAW.glob("*.csv"))

print(f"Arquivos raw encontrados: {len(arquivos_raw)}")
for arq in arquivos_raw:
    print(f"  {arq.name}")

frames = []
for arq in arquivos_raw:
    df_arq = pd.read_csv(arq, encoding="utf-8-sig", dtype=str)
    df_arq["arquivo_raw_origem"] = arq.name
    frames.append(df_arq)

df_raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal bruto acumulado: {len(df_raw)} registros")
print(f"\nDistribuição por portal:")
print(df_raw["portal"].value_counts())

Arquivos raw encontrados: 8
  rss_noticias_reais_raw_2026-05-09_16-26-39.csv
  rss_noticias_reais_raw_2026-05-10_01-58-57.csv
  rss_noticias_reais_raw_2026-05-13_00-18-59.csv
  rss_noticias_reais_raw_2026-05-16_14-03-09.csv
  rss_noticias_reais_raw_2026-05-17_03-23-31.csv
  rss_noticias_reais_raw_2026-05-17_13-27-02.csv
  rss_noticias_reais_raw_2026-05-17_13-29-11.csv
  rss_noticias_reais_raw_2026-05-19_01-04-15.csv

Total bruto acumulado: 2111 registros

Distribuição por portal:
portal
G1_POLITICA            700
FOLHA_PODER            400
BBC_BRASIL             316
CORREIO_BRAZILIENSE    120
UOL_NOTICIAS           105
AGENCIA_BRASIL          80
VEJA_POLITICA           80
METROPOLES              80
CARTACAPITAL            80
CONGRESSO_EM_FOCO       80
PODER360                70
Name: count, dtype: int64


## Limpeza de HTML e normalização de texto

In [5]:
df = df_raw.copy()

df["titulo_limpo"] = df["titulo"].apply(remover_html)
df["resumo_limpo"] = df["resumo"].apply(remover_html)


def montar_texto_e_origem(r):
    titulo = r["titulo_limpo"].strip()
    resumo = r["resumo_limpo"].strip()

    if not resumo or resumo == titulo:
        # Sem resumo ou resumo igual ao título: usar só o título
        return pd.Series({"texto_principal": titulo, "origem_texto": "titulo"})

    if len(resumo) <= RESUMO_MAX_CHARS:
        # Resumo curto o suficiente: concatenar título + resumo
        return pd.Series({"texto_principal": f"{titulo} — {resumo}", "origem_texto": "titulo_resumo"})

    # Resumo muito longo (artigo completo): usar só o título
    return pd.Series({"texto_principal": titulo, "origem_texto": "titulo"})


df[["texto_principal", "origem_texto"]] = df.apply(montar_texto_e_origem, axis=1)

# Remover registros sem título
antes = len(df)
df = df[df["titulo_limpo"].str.strip() != ""].copy()
print(f"Removidos sem título: {antes - len(df)}")

# Filtro de tamanho mínimo
antes = len(df)
df = df[df["texto_principal"].str.len() >= TEXTO_MIN_CHARS].copy()
print(f"Removidos por texto < {TEXTO_MIN_CHARS} chars: {antes - len(df)}")

# Flag de texto longo (não remove — apenas sinaliza para rastreabilidade)
df["flag_texto_longo"] = df["texto_principal"].str.len() > TEXTO_LONGO_CHARS

print(f"Registros após limpeza de texto: {len(df)}")
print(f"  Com flag_texto_longo=True : {df['flag_texto_longo'].sum()}")
print(f"\nDistribuição de origem_texto:")
print(df["origem_texto"].value_counts())

Removidos sem título: 0
Removidos por texto < 30 chars: 0
Registros após limpeza de texto: 2111
  Com flag_texto_longo=True : 0

Distribuição de origem_texto:
origem_texto
titulo_resumo    1615
titulo            496
Name: count, dtype: int64


## Filtro de relevância política (portais com categoria GERAL)

In [6]:
portais_geral = df[df["categoria"] == "GERAL"]["portal"].unique().tolist()
print(f"Portais com categoria GERAL (serão filtrados): {portais_geral}")

mask_geral = df["categoria"] == "GERAL"
mask_relevante = df["texto_principal"].apply(
    lambda t: contem_termos_politicos(t, TERMOS_POLITICOS)
)

# Manter: registros de portais políticos OU registros gerais com termos políticos
df_filtrado = df[~mask_geral | mask_relevante].copy()

removidos_filtro = len(df) - len(df_filtrado)
print(f"\nRemovidos por falta de relevância política: {removidos_filtro}")
print(f"Registros após filtro: {len(df_filtrado)}")
print(f"\nDistribuição por portal após filtro:")
print(df_filtrado["portal"].value_counts())

Portais com categoria GERAL (serão filtrados): ['BBC_BRASIL', 'UOL_NOTICIAS']

Removidos por falta de relevância política: 246
Registros após filtro: 1865

Distribuição por portal após filtro:
portal
G1_POLITICA            700
FOLHA_PODER            400
BBC_BRASIL             132
CORREIO_BRAZILIENSE    120
AGENCIA_BRASIL          80
VEJA_POLITICA           80
METROPOLES              80
CARTACAPITAL            80
CONGRESSO_EM_FOCO       80
PODER360                70
UOL_NOTICIAS            43
Name: count, dtype: int64


## Remoção de duplicatas e padronização de datas

In [7]:
df = df_filtrado.copy()

# Padronizar data de publicação
df["data_publicacao"] = df["data_publicacao"].apply(padronizar_data_rfc)

# Remover duplicatas por link (mesma notícia coletada em execuções diferentes)
antes = len(df)
df = df.drop_duplicates(subset=["link"], keep="first").copy()
print(f"Duplicatas removidas por link: {antes - len(df)}")

# Remover duplicatas remanescentes por texto_principal + portal
antes = len(df)
df = df.drop_duplicates(
    subset=["texto_principal", "portal"],
    keep="first"
).copy()
print(f"Duplicatas adicionais removidas por texto+portal: {antes - len(df)}")
print(f"Registros únicos finais: {len(df)}")

Duplicatas removidas por link: 1032
Duplicatas adicionais removidas por texto+portal: 0
Registros únicos finais: 833


## Montagem do DataFrame curated

In [8]:
df["id_registro"]      = [str(uuid.uuid4()) for _ in range(len(df))]
df["rotulo_preliminar"] = "NOTICIAS_REAIS"
df["pipeline"]          = "noticias_reais"
df["fonte"]             = df["portal"].fillna("DESCONHECIDO").str.strip()
df["tipo_conteudo"]     = "NOTICIA_JORNALISTICA"
df["url_origem"]        = df["link"].fillna("").str.strip()
df["data_curadoria"]    = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

COLUNAS_OBRIGATORIAS = [
    "id_registro",
    "texto_principal",
    "rotulo_preliminar",
    "pipeline",
    "fonte",
    "tipo_conteudo",
    "data_publicacao",
    "url_origem",
    "data_curadoria",
]

# Colunas de contexto específicas do pipeline RSS
COLUNAS_CONTEXTO = [
    "portal",
    "categoria",
    "origem_texto",       # titulo | titulo_resumo
    "flag_texto_longo",   # True se texto_principal > TEXTO_LONGO_CHARS
    "arquivo_raw_origem",
]

df_curated = df[COLUNAS_OBRIGATORIAS + COLUNAS_CONTEXTO].reset_index(drop=True)

print(f"Shape final do curated: {df_curated.shape}")
print(f"\nColunas: {list(df_curated.columns)}")
df_curated.head(3)

Shape final do curated: (833, 14)

Colunas: ['id_registro', 'texto_principal', 'rotulo_preliminar', 'pipeline', 'fonte', 'tipo_conteudo', 'data_publicacao', 'url_origem', 'data_curadoria', 'portal', 'categoria', 'origem_texto', 'flag_texto_longo', 'arquivo_raw_origem']


,id_registro,texto_principal,rotulo_preliminar,pipeline,fonte,tipo_conteudo,data_publicacao,url_origem,data_curadoria,portal,categoria,origem_texto,flag_texto_longo,arquivo_raw_origem
0,2e915113-9a19-4d76-b4d9-a6e705c8cabd,"""Ninguém respeita lambe-botas"", diz Lula sobre...",NOTICIAS_REAIS,noticias_reais,AGENCIA_BRASIL,NOTICIA_JORNALISTICA,2026-05-08,https://agenciabrasil.ebc.com.br/politica/noti...,2026-05-19 01:04:36,AGENCIA_BRASIL,POLITICA,titulo,False,rss_noticias_reais_raw_2026-05-09_16-26-39.csv
1,c62808c9-de70-45ea-ad7a-04738af6ad0b,Dosimetria: Alcolumbre promulga lei que benefi...,NOTICIAS_REAIS,noticias_reais,AGENCIA_BRASIL,NOTICIA_JORNALISTICA,2026-05-08,https://agenciabrasil.ebc.com.br/politica/noti...,2026-05-19 01:04:36,AGENCIA_BRASIL,POLITICA,titulo,False,rss_noticias_reais_raw_2026-05-09_16-26-39.csv
2,0c48469e-a9e0-44fe-a996-1f9f7dd3c2e2,Especialistas e municípios criticam PL sobre m...,NOTICIAS_REAIS,noticias_reais,AGENCIA_BRASIL,NOTICIA_JORNALISTICA,2026-05-07,https://agenciabrasil.ebc.com.br/politica/noti...,2026-05-19 01:04:36,AGENCIA_BRASIL,POLITICA,titulo,False,rss_noticias_reais_raw_2026-05-09_16-26-39.csv


## Verificação de qualidade

In [9]:
print("=== Verificação de qualidade ===")
print(f"\nTotal de registros: {len(df_curated)}")

print(f"\nValores nulos por coluna obrigatória:")
print(df_curated[COLUNAS_OBRIGATORIAS].isnull().sum())

print(f"\nDistribuição por portal:")
print(df_curated["portal"].value_counts())

print(f"\nDistribuição por categoria:")
print(df_curated["categoria"].value_counts())

print(f"\nDistribuição por origem_texto:")
print(df_curated["origem_texto"].value_counts())

print(f"\nRegistros com flag_texto_longo=True: {df_curated['flag_texto_longo'].sum()}")

lens = df_curated["texto_principal"].str.len()
print(f"\nEstatísticas de tamanho do texto_principal (chars):")
print(f"  média={lens.mean():.0f}  mediana={lens.median():.0f}  min={lens.min()}  max={lens.max()}")
print(f"  < 100c : {(lens < 100).sum()}")
print(f"  100–300c: {((lens >= 100) & (lens <= 300)).sum()}")
print(f"  300–500c: {((lens > 300) & (lens <= 500)).sum()}")
print(f"  > 500c : {(lens > 500).sum()}")

print(f"\nRegistros com data_publicacao preenchida: {(df_curated['data_publicacao'] != '').sum()}")

print(f"\nExemplos de texto_principal (5 primeiros):")
print(df_curated["texto_principal"].head(5).to_string())

=== Verificação de qualidade ===

Total de registros: 833

Valores nulos por coluna obrigatória:
id_registro          0
texto_principal      0
rotulo_preliminar    0
pipeline             0
fonte                0
tipo_conteudo        0
data_publicacao      0
url_origem           0
data_curadoria       0
dtype: int64

Distribuição por portal:
portal
G1_POLITICA            305
FOLHA_PODER            145
CORREIO_BRAZILIENSE     64
PODER360                60
VEJA_POLITICA           47
CONGRESSO_EM_FOCO       46
CARTACAPITAL            41
BBC_BRASIL              37
UOL_NOTICIAS            36
AGENCIA_BRASIL          32
METROPOLES              20
Name: count, dtype: int64

Distribuição por categoria:
categoria
POLITICA    760
GERAL        73
Name: count, dtype: int64

Distribuição por origem_texto:
origem_texto
titulo_resumo    491
titulo           342
Name: count, dtype: int64

Registros com flag_texto_longo=True: 0

Estatísticas de tamanho do texto_principal (chars):
  média=188  mediana=171

## Exportação para curated/

In [10]:
data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
caminho_saida = PASTA_CURATED / f"rss_noticias_curated_{data_agora}.csv"

df_curated.to_csv(caminho_saida, index=False, encoding="utf-8-sig")

print(f"Arquivo curated salvo em: {caminho_saida}")
print(f"Total de registros exportados: {len(df_curated)}")
print(f"Data e hora da curadoria: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo curated salvo em: ..\dados\pipeline_noticias_reais\curated\rss_noticias_curated_2026-05-19_01-04-36.csv
Total de registros exportados: 833
Data e hora da curadoria: 19/05/2026 01:04:36
